# One-Hot Encoding

For a categorical feature with **no natural order** (e.g. `fuel` type, `owner` count, `brand`) — where `OrdinalEncoder`-style integer ranks would wrongly imply one category is "more" than another — one-hot encoding is the standard fix: turn each category into its own binary (0/1) column. This notebook covers three ways to do it, plus a technique for handling columns with *many* categories.

## Setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('cars.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'cars.csv'

In [ ]:
df.head()

Checking `value_counts()` on a categorical column first is good practice — it shows exactly which categories exist and how common each is, before deciding how to encode them.

In [ ]:
df['owner'].value_counts()

## 1. OneHotEncoding using Pandas

`pd.get_dummies()` is the quickest way to one-hot encode directly on a DataFrame — pass the categorical column names via `columns=`, and it replaces each one with a set of binary columns (one per category).

In [ ]:
pd.get_dummies(df,columns=['fuel','owner'])

### A Version-Dependent Detail: `dtype`

As of **pandas 2.0**, `get_dummies()` returns columns as **`bool`** (`True`/`False`) by default, rather than the `uint8` (`0`/`1`) integers older pandas versions produced. This isn't broken — `True`/`False` behave like `1`/`0` in arithmetic and most models handle it fine — but if you specifically want integer columns (e.g. to match older code, or for a library that's picky about dtype), pass `dtype=int` explicitly.

In [ ]:
pd.get_dummies(df, columns=['fuel', 'owner'], dtype=int)


## 2. K-1 OneHotEncoding

### Why Drop One Category? — The Dummy Variable Trap

If a column has *k* categories, only *k-1* dummy columns are actually needed — the last category is fully implied when all the others are 0 (e.g. if `fuel_Diesel` and `fuel_CNG` are both 0, the fuel *must* be the dropped category). Keeping all *k* columns makes them perfectly correlated with each other, which can cause problems for models like linear/logistic regression (**multicollinearity**) — `drop_first=True` avoids this by dropping one category per column.

In [ ]:
pd.get_dummies(df,columns=['fuel','owner'],drop_first=True)

## 3. OneHotEncoding using Sklearn

`pd.get_dummies()` is quick for one-off exploration, but it has a real problem in an ML workflow: it looks at whatever categories are *actually present* in the data you give it — which can silently produce a **different set of columns** for the training set vs. the test set (e.g. if `owner` has a category in `X_test` that never appeared in `X_train`, or vice versa). `sklearn`'s `OneHotEncoder` avoids that by learning its categories once (via `.fit()`), then applying that *exact same* set of columns to any data afterward — training, test, or future production data.

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=2)

In [ ]:
X_train.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

### ⚠️ Important: `sparse=False` No Longer Works

`sparse` was deprecated in scikit-learn 1.2 and **fully removed in 1.4** — passing it now raises `TypeError: __init__() got an unexpected keyword argument 'sparse'`. It was renamed to `sparse_output` (same meaning: `False` returns a regular dense array instead of a memory-efficient sparse matrix, which is what you usually want for a small number of categories like this).

In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int32)


### Handling Categories Never Seen During Training — `handle_unknown`

By default, if `.transform()` ever encounters a category it didn't see during `.fit()` (e.g. a brand-new `owner` value that shows up in production data later), `OneHotEncoder` **raises an error**. `handle_unknown='ignore'` makes it more forgiving instead — an unseen category just gets encoded as all zeros, rather than crashing your pipeline. Worth adding for anything beyond a quick notebook experiment.

In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int32, handle_unknown='ignore')


In [ ]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']])

Correctly done here: `.fit_transform()` on the training set, then plain `.transform()` (**not** `fit_transform`) on the test set — reusing the categories learned from training only, exactly as the scaling and encoding notebooks before this one emphasized.

In [ ]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

In [ ]:
X_train_new.shape

In [ ]:
np.hstack((X_train[['brand','km_driven']].values,X_train_new))

`np.hstack()` glues the untouched numeric columns (`brand`, `km_driven`) back together with the new one-hot columns into a single array, ready for a model. Worth doing the same for `X_test_new` immediately after, so both sets end up in a consistent, model-ready shape.

In [ ]:
X_test_final = np.hstack((X_test[['brand', 'km_driven']].values, X_test_new))
X_test_final.shape


### Recovering Column Names — `get_feature_names_out()`

One real downside of the array-based workflow above: `np.hstack()` produces a plain NumPy array with no column names at all, which makes the result hard to inspect. `get_feature_names_out()` returns the actual name generated for each one-hot column (e.g. `fuel_Diesel`, `fuel_Petrol`, ...) — pairing it back with a DataFrame keeps things readable.

In [ ]:
encoded_cols = ohe.get_feature_names_out(['fuel', 'owner'])
print(encoded_cols)

pd.DataFrame(X_train_new, columns=encoded_cols).head()


## 4. OneHotEncoding with Top Categories

### The Problem With High-Cardinality Columns

`brand` likely has **dozens** of unique values (car manufacturers) — one-hot encoding all of them the same way as `fuel`/`owner` would create dozens of extra columns, most representing a handful of rare brands. That both bloats the feature space and gives a model very little data to learn from for each rare category.

In [ ]:
counts = df['brand'].value_counts()

In [ ]:
print('Unique brands:', df['brand'].nunique())
threshold = 100


The fix: only keep dummy columns for brands that appear **frequently** (above some threshold count), and lump every rarer brand into a single `'uncommon'` bucket — one column instead of dozens for the long tail of infrequent brands.

In [ ]:
df['brand'].nunique()
threshold = 100

In [ ]:
repl = counts[counts <= threshold].index

In [ ]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon')).sample(5)

## Summary

| Task | Method |
|---|---|
| Quick one-hot encoding | `pd.get_dummies(df, columns=[...])` |
| Avoid multicollinearity | `pd.get_dummies(..., drop_first=True)` |
| Consistent train/test encoding | `sklearn`'s `OneHotEncoder` — fit once, transform both |
| Dense (non-sparse) output | `sparse_output=False` *(not the removed `sparse=False`)* |
| Survive unseen categories later | `handle_unknown='ignore'` |
| Recover column names after encoding | `ohe.get_feature_names_out([...])` |
| High-cardinality columns | bucket rare categories into `'uncommon'` before encoding |

### The golden rule (same as scaling)
`.fit()` (or `.fit_transform()`) only on the training set — `.transform()` only, never re-fit, on the test set.